<a href="https://colab.research.google.com/github/dennzii/Low-Memory-Difusion-Based-Virtual-Try-On-via-Quantized-Stable-Difusion-Backbones/blob/main/concat_selfattn_sd1_5_lora_finetune6_04_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install bitsandbytes

In [ ]:
!pip install lpips
!pip install xformers

In [ ]:
import kagglehub
!pip install kagglehub

In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
import torch
from diffusers import DiffusionPipeline
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#for faster mat mul operations
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import kagglehub
import os

# Download latest version of the dataset
path = kagglehub.dataset_download("tinkukalluri/zalando-hd-resized")

# The dataset_base_path will be the path returned by kagglehub
dataset_base_path = path

print(f"✅ Dataset indirildi ve extract edildi: {dataset_base_path}")

In [ ]:
import os
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset
from PIL import Image
import torchvision.transforms as T
import numpy as np
import cv2


class VITONDataset(Dataset):

    def __init__(self, dataset_path, image_size=(512, 384)):
        self.dataset_path = dataset_path
        self.image_size = image_size

        self.file_names = sorted(os.listdir(os.path.join(dataset_path, "image")))

        self.to_tensor = T.ToTensor()
        self.resize = T.Resize(image_size, interpolation=T.InterpolationMode.BICUBIC)

    def __len__(self):
        return len(self.file_names)

    def load_rgb_tensor(self, path):
        img = Image.open(path).convert("RGB")
        img = self.resize(img)
        return self.to_tensor(img)  # (3,H,W)

    def concat_images_tensor(self, cloth, image):
        # cloth + image → width concat (same as PIL paste)
        return torch.cat([cloth, image], dim=2)  # (3, H, 2W)

    def zero_pad_left_tensor(self, mask):
        # mask: (1,H,W) → (1,H,2W)
        B, H, W = mask.shape
        pad = torch.zeros((1, H, W), device=mask.device)
        return torch.cat([pad, mask], dim=2)

    def load_agnostic_mask(self, mask_path):
        mask_img = Image.open(mask_path).convert("L")
        # PIL expects (width, height), so we swap the image_size elements
        mask_img = mask_img.resize((self.image_size[1], self.image_size[0]), resample=Image.NEAREST)

        # Değeri 0-1 aralığına çek ve threshold uygula
        mask_array = np.array(mask_img, dtype=np.float32) / 255.0
        mask_array[mask_array < 0.5] = 0.0
        mask_array[mask_array >= 0.5] = 1.0

        return torch.from_numpy(mask_array).unsqueeze(0)  # (1, H, W)

    def __getitem__(self, idx):

        file_name = self.file_names[idx]

        cloth_path = os.path.join(self.dataset_path, "cloth", file_name)
        image_path = os.path.join(self.dataset_path, "image", file_name)

        # Mask dosya ismini bulma (genelde .png veya _mask.png olabilir)
        mask_name = file_name.replace('.jpg', '.png')
        mask_path = os.path.join(self.dataset_path, "agnostic-mask", mask_name)
        if not os.path.exists(mask_path):
            mask_path = os.path.join(self.dataset_path, "agnostic-mask", file_name)
        if not os.path.exists(mask_path):
            mask_path = os.path.join(self.dataset_path, "agnostic-mask", file_name.replace('.jpg', '_mask.png'))

        # load tensors
        cloth = self.load_rgb_tensor(cloth_path)
        image = self.load_rgb_tensor(image_path)

        # concat
        concat_tensor = self.concat_images_tensor(cloth, image)

        # Hazır agnostic-mask dizininden maskeyi yükle
        mask_512 = self.load_agnostic_mask(mask_path)

        mask_pad = self.zero_pad_left_tensor(mask_512)

        return concat_tensor, mask_pad

In [ ]:
from google.colab import drive
import os

save_dir = "/content/drive/MyDrive/VTON_Project/checkpoints"
os.makedirs(save_dir, exist_ok=True)

In [ ]:


# dataset_base_path is now set by the kagglehub download in the previous cell

train_ds = VITONDataset(dataset_path=os.path.join(dataset_base_path,"train"))
test_ds = VITONDataset(dataset_path=os.path.join(dataset_base_path,"test"))

train_loader = DataLoader(train_ds,batch_size=128,shuffle=True,num_workers=8,
    pin_memory=True,
    persistent_workers=True)

test_loader = DataLoader(test_ds,batch_size=16,shuffle=False,num_workers=4)

In [ ]:
from diffusers.models.autoencoders import AutoencoderKL
from diffusers.schedulers.scheduling_ddpm import DDPMScheduler
vae = AutoencoderKL.from_pretrained("stable-diffusion-v1-5/stable-diffusion-inpainting", subfolder="vae").to(device)
vae.enable_slicing()
vae.eval()

In [ ]:
noise_scheduler = DDPMScheduler.from_pretrained("stable-diffusion-v1-5/stable-diffusion-inpainting", subfolder="scheduler")

In [ ]:
import torch
import torch.nn as nn
import gc
from diffusers import UNet2DConditionModel
from diffusers.models.attention import BasicTransformerBlock

# 1. Dummy (Boş) Cross-Attention Sınıfı
class DummyCrossAttention(nn.Module):
    def __init__(self):
        super().__init__()
        # Hiçbir parametre içermez, VRAM harcamaz.

    def forward(self, hidden_states, encoder_hidden_states=None, attention_mask=None, **kwargs):
        # İşlem yapmadan gelen veriyi doğrudan geri döndür
        return hidden_states

# 2. Kalıcı Silme Fonksiyonu
def strip_cross_attention_permanently(unet_model):
    removed_params = 0
    for name, module in unet_model.named_modules():
        if isinstance(module, BasicTransformerBlock):
            # attn2'yi (Cross-Attention) tamamen Dummy katmanla değiştir
            if hasattr(module, 'attn2') and module.attn2 is not None:
                removed_params += sum(p.numel() for p in module.attn2.parameters())
                module.attn2 = DummyCrossAttention()

            # norm2'yi (Cross-Attention öncesi normalizasyon) Identity ile değiştir
            if hasattr(module, 'norm2') and module.norm2 is not None:
                removed_params += sum(p.numel() for p in module.norm2.parameters())
                module.norm2 = nn.Identity()

    # garbage collector çalıştır ve VRAM'i temizle
    gc.collect()
    torch.cuda.empty_cache()
    print(f"Başarıyla {removed_params / 1e6:.2f}M parametre silindi ve VRAM'den temizlendi!")
    return unet_model

In [ ]:
import torch.nn as nn
from diffusers import UNet2DConditionModel

unet = UNet2DConditionModel.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-inpainting",
    subfolder="unet"
)

unet = strip_cross_attention_permanently(unet)

unet.to(device)

# önce freeze et.
for param in unet.parameters():
    param.requires_grad = False

# sadece attn1 layerlarını aç
for name, module in unet.named_modules():
    if "attn1" in name:
        for param in module.parameters():
            param.requires_grad = True

# kontrol
trainable = sum(p.numel() for p in unet.parameters() if p.requires_grad)
print("Trainable params:", trainable)

In [ ]:
unet.enable_gradient_checkpointing()
unet.enable_xformers_memory_efficient_attention()
!PYTORCH_ALLOC_CONF=expandable_segments:True

In [ ]:
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, unet.parameters()),
    lr=1e-5
)

epochs = 500

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

from tqdm import tqdm
import torch
import torch.nn.functional as F
import shutil
import os

scaler = torch.amp.GradScaler("cuda")
unet.train()

best_loss = float("inf")

for epoch in range(epochs):

    epoch_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")

    for batch in pbar:

        target_images = batch[0].to(device)
        masks = batch[1].to(device)

        # Target'ı normalize et [-1, 1]
        target_images = target_images * 2.0 - 1.0

        # Condition: Maskelenmiş görüntü (Kıyafetin kişinin üzerinden silindiği hali)
        masked_images = target_images * (1 - masks)

        # CFG Dropout: Sadece condition'ın (masked_images) sol tarafını (referans kıyafeti) sil
        # Resim [-1, 1] aralığında olduğu için silinen kısmı -1.0 (siyah) yapıyoruz
        drop_mask = torch.rand(target_images.shape[0], device=device) < 0.1
        masked_images[drop_mask, :, :, :384] = -1.0

        with torch.no_grad():
            #target latent (Buna gürültü eklenecek
            latents = vae.encode(target_images).latent_dist.sample()
            latents = latents * vae.config.scaling_factor

            # Condition latent cfg
            masked_latents = vae.encode(masked_images).latent_dist.sample()
            masked_latents = masked_latents * vae.config.scaling_factor

        masks = F.interpolate(
            masks,
            size=latents.shape[-2:],
            mode="nearest"
        )

        noise = torch.randn_like(latents)
        timesteps = torch.randint(
            0,
            noise_scheduler.config.num_train_timesteps,
            (latents.shape[0],),
            device=device
        ).long()

        noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)


        # DREAM: first pass (teacher)
        with torch.no_grad():
            with torch.amp.autocast("cuda"):
                initial_input = torch.cat(
                    [noisy_latents, masks, masked_latents],
                    dim=1
                )
                initial_noise_pred = unet(
                    initial_input,
                    timesteps,
                    encoder_hidden_states=None
                ).sample

        alpha_bar_t = noise_scheduler.alphas_cumprod[timesteps].view(-1, 1, 1, 1)
        lambda_t  = (1.0 - alpha_bar_t).sqrt() ** 10 #

        delta_eps = noise - initial_noise_pred

        # rectified noisy latent (Latent uzayında kaydırma: sqrt_one_minus_alpha çarpanı ZORUNLU)
        noisy_latents_dream = noisy_latents + lambda_t * delta_eps

        # DREAM: second pass (student)

        model_input = torch.cat(
            [noisy_latents_dream, masks, masked_latents],
            dim=1
        )

        with torch.amp.autocast("cuda"):
            noise_pred = unet(
                model_input,
                timesteps,
                encoder_hidden_states=None
            ).sample


            target = noise + lambda_t  * delta_eps

            # catvton makalesindeki gibi mse kullanırız. detaylar önemli.
            loss = F.mse_loss(noise_pred, target)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.item()
        pbar.set_postfix({"loss": loss.item()})
        #lr_scheduler.step()

    avg_loss = epoch_loss / len(train_loader)
    print(f"\nEpoch {epoch+1} Ortalama Loss: {avg_loss:.6f}")

    file_pth = f"dream1_lr_35_1e6_bicubic_epoch_{epoch}_pbc_fp32_{avg_loss:.6f}.pt"

    temp_save_path = os.path.join(
        "/content/",
        file_pth
    )

    torch.save(
        {
            "epoch": epoch,
            "unet": unet.state_dict(),
            "optimizer": optimizer.state_dict(),
            "loss": avg_loss,
        },
        temp_save_path
    )

    shutil.copy(
        temp_save_path,
        os.path.join(save_dir, file_pth)
    )


In [ ]:
from diffusers import DPMSolverMultistepScheduler
import torch.nn.functional as F
from tqdm import tqdm
import torch

# Sadece inference time için DPM++ 2M Karras scheduler
inference_scheduler = DPMSolverMultistepScheduler.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-inpainting",
    subfolder="scheduler",
    use_karras_sigmas=True
)

@torch.inference_mode()
def run_inference(batch, num_steps=25,cfg=2.5,verbose=True):

    concat_tensor, mask_tensor = batch
    concat_tensor = concat_tensor.to(device)
    mask_tensor = mask_tensor.to(device)

    # Artık direkt batch boyutunu (B, C, H, W) koruyoruz
    image_batch = concat_tensor.clone()
    mask_batch = mask_tensor.clone()

    # normalize (-1,1) vae böyle bir girdi alıyor.
    image_batch = image_batch * 2.0 - 1.0

    # her ihtimale karşı mask'ı binary hale getirelim
    mask_batch = (mask_batch > 0.5).float()

    # cond masked
    masked_image = image_batch * (1 - mask_batch)

    # Training ile birebir aynı uncond_masked_image cloth kısmını 0 yapmamız lazım ama noramlizasyon sonucu 0 = -1
    # olacağı için direkt sol tarafı -1 yapıyoruz.
    uncond_masked_image = masked_image.clone()
    uncond_masked_image[:, :, :, :384] = -1.0

    # ---------------------------------

    # VAE
    image_latents = vae.encode(image_batch).latent_dist.sample() * vae.config.scaling_factor
    masked_latents = vae.encode(masked_image).latent_dist.sample() * vae.config.scaling_factor
    uncond_masked_latents = vae.encode(uncond_masked_image).latent_dist.sample() * vae.config.scaling_factor

    mask_latent = F.interpolate(mask_batch, size=masked_latents.shape[-2:], mode="nearest")

    # diffusion
    inference_scheduler.set_timesteps(num_steps, device=device)
    timesteps = inference_scheduler.timesteps

    # Inference'ta inpainting için tam güróltü (pure noise) ile başlarız.
    latents = torch.randn_like(image_latents)

    # denoise
    # leave=False parametresi ise bar aktif olsa bile tamamlanınca ekrandan silinmesini sağlar.
    for t in tqdm(timesteps, disable=not verbose, leave=False):

        cond_input = torch.cat([latents, mask_latent, masked_latents], dim=1)
        uncond_input = torch.cat([latents, mask_latent, uncond_masked_latents], dim=1)

        with torch.autocast("cuda"):
            cond_pred = unet(cond_input, t, encoder_hidden_states=None).sample
            uncond_pred = unet(uncond_input, t, encoder_hidden_states=None).sample

        # CFG 2.5-3.5 arası en iyi kalite veriyor. catvton makalesine göre
        noise_pred = uncond_pred + cfg * (cond_pred - uncond_pred)

        latents = inference_scheduler.step(noise_pred, t, latents).prev_sample

    # decode et ve denormalize et
    latents = latents / vae.config.scaling_factor
    output = vae.decode(latents).sample
    output = (output / 2 + 0.5).clamp(0, 1)

    # blend
    original_image_01 = (image_batch / 2 + 0.5).clamp(0, 1)
    output = original_image_01 * (1 - mask_batch) + output * mask_batch

    return output.float()


In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig


unet = UNet2DConditionModel.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-inpainting",
    subfolder="unet"
)

# Cross-attention katmanlarını kalıcı olarak sil (VRAM boşalır)
unet = strip_cross_attention_permanently(unet)

# Eğittiğin checkpoint'i yükle.
# strict=False çok önemlidir! Çünkü state_dict içindeki attn2 ağırlıkları artık modelimizde yok.
ckpt_path = os.path.join(save_dir,"dream1_lr_35_1e6_bicubic_epoch_39_pbc_fp32_0.019724.pt")
ckpt = torch.load(ckpt_path, map_location="cpu")
unet.load_state_dict(ckpt["unet"], strict=False)

# Cihaza gönder
unet.to("cuda",dtype=torch.float16)
print("UNet kullanıma hazır!")

### Kullanım Örneği:
Eğittiğiniz modeli yüklerken artık şu mantığı kullanmalısınız:

In [ ]:
from torchvision.utils import save_image
from tqdm import tqdm

# Redefine test_loader with num_workers=0 to avoid multiprocessing issues with PIL.Image
# The original test_loader was defined in cell rhdlPYuNH-X7 with num_workers=4
from torch.utils.data import DataLoader # Ensure DataLoader is imported if not already in scope

test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=0)

# Farklı indeksler belirleyelim
person_idx = 103
cloth_idx = 6 # Farklı bir kıyafet için indeks

person_item = None
cloth_item = None

# DataLoader üzerinden iterasyon yaparak iki farklı batch'i alalım
for i, batch in enumerate(test_loader):
    if i == person_idx:
        person_item = batch
    if i == cloth_idx:
        cloth_item = batch

    if person_item is not None and cloth_item is not None:
        break

# Hedef kişinin tensorunu kopyalayalım
mixed_concat = person_item[0].clone()
# Sol tarafı (ilk 384 piksel, yani kıyafet kısmı) cloth_item'dan alalım
mixed_concat[:, :, :, :384] = cloth_item[0][:, :, :, :384]

# Yeni karma batch'i oluşturalım (maske kişinin kendi maskesi kalacak)
mixed_item = (mixed_concat, person_item[1])

# Çıkarımı çalıştır (artık test veri setindeki X kişisine Y kıyafeti giydirilecek)
output = run_inference(mixed_item, num_steps=25,cfg = 4.0)

# Sonucu kaydet
save_image(output, fp="/content/output.png")

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

# Load the generated image
output_image_path = "/content/output.png"
output_image = Image.open(output_image_path)

# Display the image
plt.figure(figsize=(10, 10))
plt.imshow(output_image)
plt.axis('off')
plt.title('Generated Output Image')
plt.show()

In [ ]:
import lpips
import torch
from tqdm import tqdm

# LPIPS modelini yükle (VGG ağı tabanlı)
loss_fn_vgg = lpips.LPIPS(net='vgg').to(device)
loss_fn_vgg.eval()

num_eval_samples = 30
total_lpips = 0.0
count = 0

print(f"Test seti üzerinden {num_eval_samples} örnek için LPIPS hesaplanıyor...")

for i, batch in enumerate(tqdm(test_loader, total=num_eval_samples)):
    if i >= num_eval_samples:
        break

    # Ground truth (orijinal) resmi al (concat tensor'ün sağ tarafı)
    concat_tensor = batch[0].to(device)
    _, _, H, W2 = concat_tensor.shape
    W = W2 // 2
    gt_image_01 = concat_tensor[:, :, :, W:] # [0, 1] aralığında orijinal resim

    # Çıkarımı çalıştır
    # Not: Inference fonksiyonunda tqdm olduğu için iç içe progress barlar görünebilir.
    output_01 = run_inference(batch, num_steps=25, cfg=2.5,verbose=False) # [0, 1] aralığında çıktı

    # Çıktının sadece sağ tarafını (üretilen kişi kısmını) al
    output_01 = output_01[:, :, :, W:]

    # LPIPS [-1, 1] aralığında değerler bekler, o yüzden dönüştürüyoruz
    gt_lpips = gt_image_01 * 2.0 - 1.0
    output_lpips = output_01 * 2.0 - 1.0

    # LPIPS değerini hesapla
    with torch.no_grad():
        lpips_val = loss_fn_vgg(output_lpips, gt_lpips)
        total_lpips += lpips_val.item()
        count += 1

avg_lpips = total_lpips / count
print(f"\n{count} örnek üzerinden Ortalama LPIPS Skoru: {avg_lpips:.4f}")
print("(LPIPS ne kadar düşükse, üretilen resim orijinaline o kadar çok benziyor demektir)")

In [ ]:
test_loader = DataLoader(test_ds, batch_size=16, shuffle=False, num_workers=0)

In [ ]:
import lpips
import torch
import os
from tqdm import tqdm

# LPIPS modelini yükle (zaten yüklüyse tekrar yüklemez)
if 'loss_fn_vgg' not in locals():
    loss_fn_vgg = lpips.LPIPS(net='vgg').to(device)
    loss_fn_vgg.eval()

num_eval_samples = 4
cfg_values = [3.5]
step_values = [25]

# Test edilecek modeller
checkpoints = ["dream1_bicubic_epoch_11_pbc_fp32_0.019168.pt"]

results = []
total_combinations = len(checkpoints) * len(cfg_values) * len(step_values)
print(f"Grid Search Başlıyor: {num_eval_samples} örnek üzerinde toplam {total_combinations} kombinasyon test edilecek.")

for ckpt_name in checkpoints:
    print(f"\n=======================================================")
    print(f"Model Yükleniyor: {ckpt_name}")
    print(f"=======================================================")

    # İlgili checkpoint'i UNet'e yükle (strict=False önemli, çünkü cross_attn yok)
    ckpt_path = os.path.join(save_dir, ckpt_name)
    ckpt = torch.load(ckpt_path, map_location="cpu")
    unet.load_state_dict(ckpt["unet"], strict=False)
    unet.to("cuda")#az sonra bunu float32 yapcam

    for cfg in cfg_values:
        for steps in step_values:
            print(f"\n--- Test: Model={ckpt_name[:12]}..., CFG={cfg}, Steps={steps} ---")
            total_lpips = 0.0
            count = 0

            for i, batch in enumerate(tqdm(test_loader, total=num_eval_samples, leave=False)):
                if i >= num_eval_samples:
                    break

                concat_tensor = batch[0].to(device)
                _, _, H, W2 = concat_tensor.shape
                W = W2 // 2
                gt_image_01 = concat_tensor[:, :, :, W:]

                # Inference'ı çalıştır
                output_01 = run_inference(batch, num_steps=steps, cfg=cfg, verbose=False)
                output_01 = output_01[:, :, :, W:]

                # LPIPS için [-1, 1] aralığına getir
                gt_lpips = gt_image_01 * 2.0 - 1.0
                output_lpips = output_01 * 2.0 - 1.0

                with torch.no_grad():
                    lpips_val = loss_fn_vgg(output_lpips, gt_lpips)
                    total_lpips += lpips_val.sum().item()
                    count += lpips_val.size(0)

            avg_lpips = total_lpips / count

            results.append({
                "model": ckpt_name,
                "cfg": cfg,
                "steps": steps,
                "lpips": avg_lpips
            })
            print(f"Sonuç -> LPIPS: {avg_lpips:.4f}")

# Sonuçları LPIPS değerine göre sırala (Düşük olması iyidir)
print("\n=== Grid Search Sonuçları Özeti (En İyiden En Kötüye) ===")
results_sorted = sorted(results, key=lambda x: x["lpips"])
for i, res in enumerate(results_sorted):
    print(f"{i+1:02d}. LPIPS: {res['lpips']:.4f} | Model: {res['model']} | CFG: {res['cfg']} | Steps: {res['steps']}")


In [ ]:
ckpts = os.listdir(save_dir)

In [ ]:
ckpts